# Question 4: Optimising pre-processing and feature extraction (50 marks)

In [1]:
import csv                               # csv reader
from sklearn.svm import LinearSVC
from nltk.classify import SklearnClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_fscore_support # to report on precision and recall
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


#ntlk download for stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/carlotacortes/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
## Global variables for experiment configuration. GO CHANGING BASED ON EXPERIMENT true or false. AND UNIGRAM OR BIGRAM (1,1) or (1,2)
## FUNCTIONS CONFIGURABLE FOR OPTIMIZATION, so no duplicating code
APPLY_LOWERCASE = True
REMOVE_STOPWORDS = False
HANDLE_NEGATION = False
USE_TFIDF = True
NGRAM_RANGE = (1,2) #(1,1) or (1,2)
SVM_c = 30 #0.01, 0.1, 1, 10, 20
SVM_balanced = False

In [3]:
def load_data(path):
    """Load data from a tab-separated file and append it to raw_data."""
    with open(path) as f:
        reader = csv.reader(f, delimiter='\t')
        for line in reader:
            (label, text) = parse_data_line(line)
            raw_data.append((text, label))

def split_and_preprocess_data(percentage):
    """Split the data between train_data and test_data according to the percentage
    and performs the preprocessing."""
    num_samples = len(raw_data)
    num_training_samples = int((percentage * num_samples))
    for (text, label) in raw_data[:num_training_samples]:
        # preprocessing with options
        #gloabl variables for experiment configuration. In preprocess it handles if you want lowercase, stopwords or handle negation
        tokens = pre_process(text, lowercase=APPLY_LOWERCASE, remove_stopwords=REMOVE_STOPWORDS, handle_negation=HANDLE_NEGATION)
        train_data.append((to_feature_vector(tokens),label, text))
    for (text, label) in raw_data[num_training_samples:]:
        tokens = pre_process(text, lowercase=APPLY_LOWERCASE, remove_stopwords=REMOVE_STOPWORDS, handle_negation=HANDLE_NEGATION)
        test_data.append((to_feature_vector(tokens),label, text))
        
def parse_data_line(data_line):
    """Return a tuple of the label as just FAKE or REAL and the statement"""
    #not fake or real, its negative or positive for sentiment analysis!!
    return (data_line[1], data_line[2])

# Preprocess

In [4]:
### CHANGES FOR EXPERIMENT CONFIGURATION: lowercase, stopwords, negation handling!!!
def pre_process(text, lowercase=False, remove_stopwords=False, handle_negation=False):
    """Return a list of tokens based on pre-processing config"""
    
    if lowercase:
        text = text.lower()
    
    tokens = text.split() #simple tokenization before
        
    if remove_stopwords:
        #nltk stopwords
        from nltk.corpus import stopwords
        stop_words = set(stopwords.words('english'))
        tokens = [t for t in tokens if t not in stop_words]
    
    if handle_negation:
        #simple negation handling: append _NEG to words following a negation word until the next punctuation
        negation_words = {"not", "no", "never", "n't"}
        punctuation = {".", "!", "?", ",", ";", ":"}
        negated = False
        new_tokens = []
        for token in tokens:
            token = token.lower() #make sure lower because negation words are in lowercase
            if token in negation_words:
                negated = True
                new_tokens.append(token)
            elif token in punctuation:
                negated = False
                new_tokens.append(token)
            else:
                if negated:
                    new_tokens.append(token + "_NEG")
                else:
                    new_tokens.append(token)
        tokens = new_tokens
        
    return tokens

## Preprocess test print:

In [5]:
# debug test
#text = "RT @colonelkickhead: Another bloody instant restaurant week?!?! Seriously! They just jumped the shark riding two other sharks powered by sh…"
#pre_process(text, lowercase=APPLY_LOWERCASE, remove_stopwords=REMOVE_STOPWORDS, handle_negation=HANDLE_NEGATION)

# Feature Extraction

In [6]:
##to_feature_vector for raw counts
global_feature_dict = {} # A global dictionary of features

def to_feature_vector(tokens):
    #given a preprocessed list of tokens, return a feature vector as a dictionary of features and weights
    feature_vector = {}
    #iterate through tokens and add weights/counts to feature_vector
    for token in tokens:
        feature_vector[token] = feature_vector.get(token, 0) + 1
        global_feature_dict[token] = global_feature_dict.get(token, 0) + 1 #update global feature dict. not needed for linearSVC but keep track of it anyway
    return feature_vector

#  Train_classifier, Predict_labels

In [7]:
# TRAINING AND VALIDATING OUR CLASSIFIER

def train_classifier(data):
    
    #SVM config parameters with global variable
    if SVM_balanced:
        class_weight = 'balanced'
    else:
        class_weight = None    
    
    if USE_TFIDF:
        X_train = [text for (features, label, text) in data] #not features, raw data
        y_train = [label for (features, label, text) in data]
        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(ngram_range=NGRAM_RANGE)), #TfidVectorizer already preprocess
            ('svc', LinearSVC(C=SVM_c, class_weight=class_weight, max_iter=10000)) #c= SVM_c, class_weight= balanced or not, high max_iter to ensure convergence
        ])
        model = pipeline.fit(X_train, y_train) #.fit() to train
        return model

    else:
        if NGRAM_RANGE != (1,1): #if bigrams
            X_train = [text for (features, label, text) in data] #not features, raw data
            y_train = [label for (features, label, text) in data]
            pipeline = Pipeline([('count', CountVectorizer(ngram_range=NGRAM_RANGE)),
                                ('svc', LinearSVC(C=SVM_c, class_weight=class_weight, max_iter=10000))])
            model = pipeline.fit(X_train, y_train) #.fit() to train
            return model
        else: #count data with no n-grams
            #ignore raw data. as linearSVC expects features and labels only. 2 max values to unpack. 
            train_data = [(features, label) for (features, label, text) in data]
            pipeline =  Pipeline([('svc', LinearSVC())])
            return SklearnClassifier(pipeline).train(train_data)

In [8]:
# PREDICTING LABELS GIVEN A CLASSIFIER

def predict_labels(samples, classifier):
    """Assuming preprocessed samples, return their predicted labels from the classifier model."""
    if USE_TFIDF or NGRAM_RANGE != (1,1):
        return classifier.predict([text for text in samples])
    else:
        return classifier.classify_many(samples)

def predict_label_from_raw(sample, classifier):
    """Assuming raw text, return its predicted label from the classifier model."""
    tokens = pre_process(sample, lowercase=APPLY_LOWERCASE, remove_stopwords=REMOVE_STOPWORDS, handle_negation=HANDLE_NEGATION) #preprocess with glb variables
    return classifier.classify(to_feature_vector(tokens))

# Cross-validation

In [9]:
#solution
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


def cross_validate(dataset, folds):
    results = []
    fold_size = int(len(dataset)/folds) + 1
    
    np.random.shuffle(dataset)  # shuffle the dataset before splitting into folds
    
    for i in range(0,len(dataset),int(fold_size)):
        # insert code here that trains and tests on the 10 folds of data in the dataset
        #print("Fold start on items %d - %d" % (i, i+fold_size))
        # FILL IN THE METHOD HERE
        
        #split into train and vald folds
        val_fold = dataset[i:i+fold_size]
        train_fold = dataset[:i] + dataset[i+fold_size:]
        
        #train classifier on train folds
        classifier = train_classifier(train_fold)
        
        true_labels = [label for (features, label, text) in val_fold] #true labels val fold
        
        #prepare val data depending on feature extraction method. tdf-idf or n-grams: raw text. Whereas counts: features. 
        if USE_TFIDF or NGRAM_RANGE != (1,1):
            val_samples = [text for (features, label, text) in val_fold]
            predicted_labels= predict_labels(val_samples, classifier)
        else: #count data with no n-grams
            val_features = [features for (features, label, text) in val_fold]
            predicted_labels = predict_labels(val_features, classifier)
        
        #calculate metrics
        precision, recall, f1_score, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='weighted')
        accuracy = accuracy_score(true_labels, predicted_labels)
        
        #append results of this fold
        results.append({'precision': precision, 'recall': recall, 'f1_score': f1_score, 'accuracy': accuracy})
        
    #average results across folds
    cv_results = {}
    for metric in results[0].keys():
        cv_results[metric] = np.mean([result[metric] for result in results])
    
    return cv_results

# MAIN

In [10]:
# MAIN

# loading reviews
# initialize global lists that will be appended to by the methods below
raw_data = []          # the filtered data from the dataset file
train_data = []        # the pre-processed training data as a percentage of the total dataset
test_data = []         # the pre-processed test data as a percentage of the total dataset


# references to the data files
data_file_path = 'sentiment-dataset.tsv'

# Do the actual stuff (i.e. call the functions we've made)
# We parse the dataset and put it in a raw data list
print("Now %d rawData, %d trainData, %d testData" % (len(raw_data), len(train_data), len(test_data)),
      "Preparing the dataset...",sep='\n')

load_data(data_file_path) 

# We split the raw dataset into a set of training data and a set of test data (80/20)
# You do the cross validation on the 80% (training data)
# We print the number of training samples and the number of features before the split
print("Now %d rawData, %d trainData, %d testData" % (len(raw_data), len(train_data), len(test_data)),
      "Preparing training and test data...",sep='\n')

split_and_preprocess_data(0.8)

# We print the number of training samples and the number of features after the split
print("After split, %d rawData, %d trainData, %d testData" % (len(raw_data), len(train_data), len(test_data)),
      "Training Samples: ", len(train_data), "Features: ", len(global_feature_dict), sep='\n')

cross_validate(train_data, 10)  # will work and output overall performance of p, r, f-score when cv implemented

Now 0 rawData, 0 trainData, 0 testData
Preparing the dataset...
Now 33540 rawData, 0 trainData, 0 testData
Preparing training and test data...
After split, 33540 rawData, 26832 trainData, 6708 testData
Training Samples: 
26832
Features: 
91358


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/Applications/anacon

{'precision': 0.8683645757233232,
 'recall': 0.8695201481847811,
 'f1_score': 0.8680221381039607,
 'accuracy': 0.8695201481847811}

# FINAL TESTING:

In [11]:
# Finally, check the accuracy of your classifier by training on all the traning data
# and testing on the test set
# Will only work once all functions are complete
functions_complete = True  # set to True once you're happy with your methods for cross val
if functions_complete:
    print(test_data[0])   # have a look at the first test data instance
    classifier = train_classifier(train_data)  # train the classifier
    test_true = [t[1] for t in test_data]   # get the ground-truth labels from the data
    
    #added for test samples for td-idf and n-grams in the case of my final config, which needs raw data instead of features
    test_samples = [text for (features, label, text) in test_data]
    #test_samples = [features for (features, label, text) in test_data]  # baseline
    
    test_pred = predict_labels(test_samples, classifier)  # classify the test data to get predicted labels
    final_scores = precision_recall_fscore_support(test_true, test_pred, average='weighted') # evaluate
    print("Done training!")
    print("Precision: %f\nRecall: %f\nF Score:%f" % final_scores[:3])

({'tomorrow': 1, "we'll": 2, 'release': 1, 'our': 2, '58th': 1, 'episode': 1, 'of': 1, '#hsonair': 1, 'profiling': 1, 'very': 1, 'own': 1, '@alissadossantos': 1, '!': 1, 'talk': 1, 'about': 1, 'storytelling': 1, 'and': 1, 'beyonce!': 1}, 'positive', "Tomorrow we'll release our 58th episode of #HSonAir profiling our very own @AlissaDosSantos ! We'll talk about storytelling and Beyonce!")


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Done training!
Precision: 0.866494
Recall: 0.867770
F Score:0.866215
